# vLLM inference — pipeline integration test

Uses `OpenAICompatibleClient` (same as the real pipeline) with credentials
from `.env` pointing at the RunPod endpoint.

In [1]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))
from api.config import get_settings

from dotenv import load_dotenv
load_dotenv("../.env")

# OPENAI_MODEL is empty in .env — override with the --served-model-name from serve_vllm.sh
# os.environ.setdefault("OPENAI_MODEL", "Qwen/Qwen3-4B")

from app.api_client.openai_compatible_client import OpenAICompatibleClient
from app.api_client.base import Message

client = OpenAICompatibleClient(
    model=os.getenv("OPENAI_MODEL", "Qwen/Qwen3-4B"),
    api_key=os.getenv("VLLM_API_KEY"),
    base_url=os.getenv("vllm_base_url"),
)

print(f"base_url : {client._client.base_url}")
print(f"model    : {client.model}")

base_url : https://kros53pn2i1l7x-8017.proxy.runpod.net/v1/
model    : Qwen/Qwen3-4B


In [4]:
# --- 1. Confirm the model is reachable -----------------------------------
models = client._client.models.list()
print("Served models:", [m.id for m in models.data])

Served models: ['Qwen/Qwen3-4B']


In [ ]:
# --- 2. Plain text call + reasoning extraction ---------------------------
result = client.call([
    Message(role="system", content="You are a helpful assistant."),
    Message(role="user",   content="What is 7 * 8? Just give the number. /think"),
])

print("Answer:", result.content)
print("Reasoning:", result.reasoning[:300] if result.reasoning else "(none)")

Answer   : 

56
Reasoning: 
Okay, the user is asking what 7 multiplied by 8 is, and they just want the number. Let me think. I know that 7 times 8 is a basic multiplication fact. Let me recall... 7 times 8. Hmm, 7 times 10 is 70, so subtract 7 times 2, which is 14. So 70 minus 14 is 56. Wait, that's right. Alternatively, I ca


In [6]:
# --- 3. Structured output — same schema the generator uses ---------------
from app.generator.prompts import (
    GENERATOR_SYSTEM_PROMPT,
    GENERATE_USER_TEMPLATE,
    HypothesesResponse,
)
from app.models import RetrieverResult

mock_retriever_output = RetrieverResult(
    content=(
        "Evidence summary:\n"
        "- Induction heads form during a sharp phase transition in transformer training (Olsson et al. 2022).\n"
        "- Apparent emergent abilities may be an artefact of discontinuous metrics (Schaeffer et al. 2023).\n"
        "- Model scale correlates with sudden capability gains across diverse tasks (Wei et al. 2022)."
    )
)

question = "Why do large language models exhibit emergent abilities at scale?"

result = client.call(
    [
        Message(role="system", content=GENERATOR_SYSTEM_PROMPT),
        Message(role="user",   content=GENERATE_USER_TEMPLATE.format(
            prompt=question,
            retriever_output=mock_retriever_output.content,
        )),
    ],
    response_schema=HypothesesResponse,
)

print("Reasoning:\n", result.reasoning[:500] if result.reasoning else "(none)", "\n")
print("Hypotheses:")
for i, h in enumerate(result.content.hypotheses, 1):
    print(f"  {i}. {h}")

Reasoning:
 
Okay, let's tackle this. The user is asking why large language models exhibit emergent abilities at scale. The evidence given includes things like induction heads forming during a phase transition, emergent abilities possibly being an artifact of discontinuous metrics, and model scale correlating with sudden capability gains.

First, I need to make sure each hypothesis is clear and precise. For example, the first one mentions phase transitions and induction heads. That's from Olsson et al. 2022 

Hypotheses:
  1. Large language models exhibit emergent abilities at scale due to a phase transition in training dynamics, where the formation of induction heads during this transition enables the model to suddenly acquire novel pattern-induction capabilities that are not present in smaller models.
  2. The apparent emergent abilities of large language models are an artifact of discontinuous metric evaluation, as sudden capability gains observed at scale may result from metric thr

## Embedder check

In [2]:
import yaml
from openai import OpenAI

with open("../config/app/config.yaml") as f:
    config = yaml.safe_load(f)

weaviate_cfg = config.get("search", {}).get("weaviate", {})
embedding_model   = weaviate_cfg.get("embedding_model", "Qwen/Qwen3-Embedding-4B")

# .env fields take priority over config.yaml (pydantic-settings double-underscore convention)
embedding_url     = os.environ.get("search__api_weaviate__embedding_url") or weaviate_cfg.get("embedding_url")
embedding_host    = os.environ.get("search__api_weaviate__embedding_host") or weaviate_cfg.get("embedding_host", "localhost")
embedding_port    = os.environ.get("search__api_weaviate__embedding_port") or weaviate_cfg.get("embedding_port", 8018)
embedding_api_key = os.environ.get("search__api_weaviate__embedding_api_key") or weaviate_cfg.get("embedding_api_key", "dummy")

embed_client = OpenAI(
    api_key=embedding_api_key,
    base_url=f"{embedding_url}/v1" if embedding_url else f"http://{embedding_host}:{embedding_port}/v1",
)

print(f"Embedder base_url : {embed_client.base_url}")
print(f"Embedder model    : {embedding_model}")

Embedder base_url : https://kros53pn2i1l7x-8018.proxy.runpod.net/v1/
Embedder model    : Qwen/Qwen3-Embedding-4B


In [3]:
# --- confirm endpoint is reachable ---
models = embed_client.models.list()
print("Served models     :", [m.id for m in models.data])

Served models     : ['Qwen/Qwen3-Embedding-4B']


In [7]:
import numpy as np
# --- embed a test sentence and inspect the result -----------------------
test_text = "Attention mechanisms allow transformers to focus on relevant parts of the input."
test_text_2 = "Transformers use attention to weigh the importance of different input tokens."

response = embed_client.embeddings.create(
    model=embedding_model,
    input=[test_text.strip(), test_text_2.strip()],
)

embedding = np.array(response.data[0].embedding)
embedding_2 = np.array(response.data[1].embedding)
print(f"Input       : {test_text}")
print(f"Dimensions  : {embedding.shape[0]}")
print(f"First 8 dims: {[round(v, 6) for v in embedding[:8]]}")
print(f"Norm        : {np.linalg.norm(embedding):.6f}")
print(f"Similarity with itself: {np.dot(embedding, embedding) / (np.linalg.norm(embedding) ** 2):.6f}")
print(f"Similarity with second embedding: {np.dot(embedding, embedding_2) / (np.linalg.norm(embedding) * np.linalg.norm(embedding_2)):.6f}")

Input       : Attention mechanisms allow transformers to focus on relevant parts of the input.
Dimensions  : 2560
First 8 dims: [np.float64(-0.000259), np.float64(0.028537), np.float64(0.06125), np.float64(-0.002364), np.float64(-0.001341), np.float64(0.075634), np.float64(0.053594), np.float64(-0.044777)]
Norm        : 1.000000
Similarity with itself: 1.000000
Similarity with second embedding: 0.939640
